# Auto-Encoders

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import os
import torch.optim as optim

# Early Stopping

In [8]:
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = val_loss

        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()

        elif score > self.best_score:
            self.counter += 1
            
            if self.counter >= self.patience:
                self.early_stop = True
                
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def reset(self):
        self.__init__()

    def load_best_model(self, model):
        model.load_state_dict(self.best_model_state)

# Base Autoencoder Architecture

In [12]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()

        # Encoder
        self.encoder = nn.sequential(
            nn.linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, 1)
        )

        # Decoder
        self.decoder = nn.sequential(
            nn.Linear(1, hidden_dim), 
            nn.ReLU(inplace=True), 
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return z, x_hat

# Trianing Loop

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoEncoder()

# Hyper Paramaters
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
early_stopping = EarlyStopping(patience=5)

TypeError: AutoEncoder.__init__() missing 1 required positional argument: 'input_dim'

In [ ]:
# Training Function
def train_epoch(progress_bar):

    loss = 0
    accuracy = 0
    samples = 0

    for X, Y in progress_bar:
        X, Y = X.to(device), Y.to(device)

        # Forward pass
        optimizer.zero_grad() 
        logits = model(X).squeeze(1)
        Y_float = Y.float()
        batch_loss = criterion(logits, Y_float)

        # Backwords pass
        batch_loss.backward()
        optimizer.step()


        # loss + accuracy calculations
        with torch.no_grad():
            size = Y.size(0)
            loss += batch_loss.item()*size
            prob = torch.sigmoid(logits)
            pred = (prob > 0.5).float()
            accuracy += (pred == Y_float).sum()
            samples += size
            progress_bar.set_postfix()
    
    avg_loss= loss/samples
    percent = accuracy/samples*100
    print(f'    Average Training Loss: {avg_loss:.4f}')
    print(f'    Training Accuracy: {percent:.2f}%')
    
    return avg_loss, percent